# Small-model dev run on a Colab T4 (draft, untested)

Runs our 36-task dev subset with E4B or E2B on a free Colab T4, with the same harness, environment pins and
gold-patch check as the RunPod runs (`docs/GPU_RUNS.md` sections 8-12). Reference for the vLLM-on-T4 part:
`docs/reference/colab/beam_run_inference_vllm.ipynb` (Apache Beam's notebook).

**Open question this notebook answers first:** does vLLM 0.19.1 start the 4-bit Gemma 4 small models on a T4
(compute 7.5, no bf16)? Step 3 runs ONE task as a probe, with `ER_VLLM_DTYPE=half`. Do not start the full run
before the probe worked.

What you need: a T4 runtime (Runtime > Change runtime type), a Colab secret `KAGGLE_API_TOKEN`
(the key icon in the sidebar, notebook access on; never paste it into a cell), and `data.tar` plus
`dev_subset.json` on your Google Drive (see step 1). The competition data stays in your own Drive and Colab
session: delete both afterwards.

## 1. Make the data file on your own machine

From the repo root, with the competition data downloaded (`python scripts/fetch_data.py --snapshots scored`
plus the seven dev snapshots that are not "scored"; see `docs/GPU_RUNS.md` section 8). The pack list is
`tasks.jsonl`, `graphs`, `embeddings`, `wheels` and the 36 `snapshots/<id>.tgz` of the dev subset:

```
tar -cf data.tar -C data/competition -T pack_list.txt     # about 7.8 GiB
```

Upload `data.tar` and `data/dev_subset.json` to `MyDrive/gemma/` in Google Drive (keep them private).

In [ ]:
# Settings
MODEL = "gemma-4-e2b-it-qat-w4a16-ct"  # or gemma-4-e4b-it-qat-w4a16-ct
BUNDLE = "e2b_v1"  # experiments/bundles/<name>: e2b_v1, e4b_v1, or a later *_v2
DRIVE_DIR = "/content/drive/MyDrive/gemma"
N_PROBE = 1  # tasks in the probe
CONTROLS = "fastapi_11194,fastapi_14077,requests_6644"  # solved by every Kaggle run; the env gate

In [ ]:
# Check the runtime
import subprocess
import sys

print(sys.version)
assert sys.version_info[:2] == (3, 12), (
    "the wheelhouse is built for Python 3.12; this runtime has " + sys.version[:4]
)
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,compute_cap", "--format=csv"],
        capture_output=True,
        text=True,
    ).stdout
)

In [ ]:
# Kaggle token from Colab secrets (never printed), the repo, and the data from Drive
import os
import subprocess

from google.colab import drive, userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
drive.mount("/content/drive")
if not os.path.isdir("/content/repo"):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/leejianrong/gemma-4-dev-agent",
            "/content/repo",
        ],
        check=True,
    )
os.makedirs("/content/data", exist_ok=True)
if not os.path.exists("/content/data/tasks.jsonl"):
    subprocess.run(["tar", "-xf", f"{DRIVE_DIR}/data.tar", "-C", "/content/data"], check=True)
print(
    subprocess.run(
        "ls /content/data | head; du -sh /content/data", shell=True, capture_output=True, text=True
    ).stdout
)

In [ ]:
# The kaggle CLI (the model and wheelhouse come from Kaggle, as on the pods)
!pip install -q kaggle==2.2.4 six

## 3. Probe: one task, with the T4 setting

`gpu_run.py` does what the pods do: environment pins, the three-control gold-patch check, model and
wheelhouse download, the pinned install (about 13 minutes; it replaces Colab's torch, so if Colab asks to
restart the runtime, do so and rerun from the settings cell), vLLM, the task. `ER_VLLM_DTYPE=half` is the T4
setting from the Beam notebook. Watch the log for `vLLM up after` and the `TASK 1/1` line.

In [ ]:
import json
import os
import shutil
import subprocess

ids = json.load(open(f"{DRIVE_DIR}/dev_subset.json"))["ids"]


def run_eval(tasks, tag):
    env = {
        **os.environ,
        "SUBMISSION_DIR": f"experiments/bundles/{BUNDLE}",
        "EVAL_TASKS": ",".join(tasks),
        "EVAL_RUN_ID": f"colab-{tag}",
        "EVAL_SEED": "1",
        "EVAL_MAX_MIN": "480",
        "DATA_DIR": "/content/data",
        "GPU_WORK": "/content/work",
        "GPU_MODEL": MODEL,
        "ER_VLLM_DTYPE": "half",
        "CONTROL_IDS": CONTROLS,
    }
    subprocess.run(
        ["python3", "kaggle/build_kernel.py", "colab", "eval"],
        cwd="/content/repo",
        env=env,
        check=True,
    )
    shutil.copy(
        "/content/repo/kaggle/eval_kernel/kernel.py", "/content/repo/pod/kernel.py"
    )  # gpu_run.py runs it
    log = open(f"/content/run_{tag}.log", "w")
    return subprocess.Popen(
        ["python3", "-u", "pod/gpu_run.py", "/content/data", f"/content/out_{tag}"],
        cwd="/content/repo",
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
    )


probe = run_eval(ids[:N_PROBE], "probe")
print("started; tail the log in the next cell")

In [ ]:
# Tail the probe log (rerun until it ends)
!tail -n 25 /content/run_probe.log

## 4. Full run (only if the probe worked)

About 2 hours on a faster card; a T4 may be slower and a free session can disconnect. The run writes to
`/content/out_full`. When it ends, copy `summary.json` and `task_results.csv` to Drive and bring them back;
share only the aggregates (the competition data rules forbid sharing per-task data).

In [ ]:
full = run_eval(ids, "full")
print("started")

In [ ]:
!tail -n 15 /content/run_full.log

In [ ]:
# Aggregates only
import json

s = json.load(open("/content/out_full/summary.json"))
print({k: s[k] for k in ("tasks", "resolved", "resolution_rate", "by_status", "by_repo", "wall_s")})
!cp /content/out_full/summary.json /content/out_full/task_results.csv {DRIVE_DIR}/

## Clean up

Delete `data.tar` from Drive and `/content/data` when done (`!rm -rf /content/data /content/work`).